# Commodity Price Merge

Takes the existing mastersets (DK1, NO2, ES) and adds the 3 commodity price columns. Output goes to data/processed/enriched so we dont overwrite the original files.

In [ ]:
#pip install yfinance pandas matplotlib
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
MASTERSET_DIR = Path('../../data/processed')
OUTPUT_DIR    = Path('../../data/processed/enriched')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ZONES = ['DK1', 'NO2', 'ES']

START = "2022-01-01"
END   = pd.Timestamp.today().strftime("%Y-%m-%d")

Downloading commodity prices:

In [ ]:
TICKERS = {
    "ttf_gas_eur_mwh"   : "TTF=F",  # gas price
    "co2_eua_eur_tonne" : "CO2.L",  # co2 allowance price
    "brent_usd_bbl"     : "BZ=F",   # oil price
}

raw_data = {}
for col_name, ticker in TICKERS.items():
    print(f"Downloading {ticker} ({col_name})...")
    df = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
    if df.empty:
        print(f"  No data returned for {ticker}")
    else:
        raw_data[col_name] = df["Close"].squeeze()
        print(f"  {len(df)} rows | {df.index[0].date()} to {df.index[-1].date()}")

commodities = pd.DataFrame(raw_data)
commodities.index.name = "date"
commodities.tail(3)

Applying D-2 lag (Section 4.2.4 of the project plan):

In [ ]:
def apply_d2_lag(df):
    full_calendar = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")
    df_daily = df.reindex(full_calendar).ffill()
    df_lagged = df_daily.shift(2)
    df_lagged.index.name = "date"
    return df_lagged

commodities_lagged = apply_d2_lag(commodities)
commodities_lagged.tail(3)

Merging into the mastersets:

In [ ]:
for zone in ZONES:
    print(f"\nProcessing {zone}...")
    
    masterset_path = MASTERSET_DIR / f"{zone}_masterset.csv"
    if not masterset_path.exists():
        print(f"  Not found: {masterset_path}")
        continue
    
    df = pd.read_csv(masterset_path, index_col=0, parse_dates=True)
    print(f"  Loaded: {df.shape} | {df.index.min()} to {df.index.max()}")
    
    df_dates = pd.to_datetime(df.index, utc=True).normalize().tz_localize(None).date
    date_index = pd.DatetimeIndex([pd.Timestamp(d) for d in df_dates])
    
    commodity_values = commodities_lagged.reindex(date_index)
    commodity_values.index = df.index
    
    df_enriched = df.join(commodity_values, how='left')
    
    output_path = OUTPUT_DIR / f"{zone}_masterset_enriched.csv"
    df_enriched.to_csv(output_path)
    print(f"  Saved: {output_path.name}")

print("\nDone.")

Quick check - all 24 hours of the same day should have the same commodity value:

In [ ]:
df_check = pd.read_csv(OUTPUT_DIR / "DK1_masterset_enriched.csv", index_col=0, parse_dates=True)
sample_day = df_check.loc["2023-06-15 00:00:00+01:00":"2023-06-15 23:00:00+01:00"]
print(sample_day[['price_eur_mwh', 'ttf_gas_eur_mwh', 'co2_eua_eur_tonne', 'brent_usd_bbl']].to_string())

Missing values check:

In [ ]:
for zone in ZONES:
    df = pd.read_csv(OUTPUT_DIR / f"{zone}_masterset_enriched.csv", index_col=0, parse_dates=True)
    print(f"\n{zone}:")
    print(df.isna().sum().to_string())

Plot of electricity price vs commodity prices for DK1:

In [ ]:
df_plot = pd.read_csv(OUTPUT_DIR / "DK1_masterset_enriched.csv", index_col=0, parse_dates=True)
df_plot.index = pd.to_datetime(df_plot.index, utc=True)
df_daily = df_plot[['price_eur_mwh', 'ttf_gas_eur_mwh', 'co2_eua_eur_tonne']].resample('D').mean()

fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True)

axes[0].plot(df_daily.index, df_daily['price_eur_mwh'], color='steelblue', linewidth=0.9)
axes[0].set_title('DK1 Day-Ahead Electricity Price (EUR/MWh)')
axes[0].set_ylabel('EUR/MWh')

axes[1].plot(df_daily.index, df_daily['ttf_gas_eur_mwh'], color='darkorange', linewidth=0.9)
axes[1].set_title('TTF Gas Price (EUR/MWh)')
axes[1].set_ylabel('EUR/MWh')

axes[2].plot(df_daily.index, df_daily['co2_eua_eur_tonne'], color='tomato', linewidth=0.9)
axes[2].set_title('EU ETS CO2 Allowance Price (EUR/tonne)')
axes[2].set_ylabel('EUR/tonne')

plt.tight_layout()
plt.show()